<a href="https://colab.research.google.com/github/bingxiaochen/ST554/blob/main/ST_554_HW7.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Bing Chen

ST 554 HW7

03/29/2026

# Set up Envrionment

In [60]:
from sklearn.model_selection import train_test_split, cross_validate
from sklearn.preprocessing import PolynomialFeatures
from sklearn.linear_model import LinearRegression, LassoCV, RidgeCV, ElasticNetCV, LogisticRegression, LogisticRegressionCV
from sklearn.metrics import mean_squared_error, mean_absolute_error
import numpy as np
import pandas as pd


# Read in Data

Two data files `winequality-red.csv` and `winequality-white.csv` files is read in from UCI machine learning repository site which contains information about read and white wine quality. We'll combine these two dataset and create a new column `Type` for represents the type of wine.

In [3]:


red = pd.read_csv("winequality-red.csv")
white = pd.read_csv("winequality-white.csv")

wine = pd.concat([
    pd.read_csv("winequality-red.csv", sep=';').assign(Type='Red'),
    pd.read_csv("winequality-white.csv", sep=';').assign(Type='White')
], ignore_index=True)

wine.head()

,fixed acidity,volatile acidity,citric acid,residual sugar,chlorides,free sulfur dioxide,total sulfur dioxide,density,pH,sulphates,alcohol,quality,Type
0,7.4,0.70,0.00,1.9,0.076,11.0,34.0,0.9978,3.51,0.56,9.4,5,Red
1,7.8,0.88,0.00,2.6,0.098,25.0,67.0,0.9968,3.20,0.68,9.8,5,Red
2,7.8,0.76,0.04,2.3,0.092,15.0,54.0,0.9970,3.26,0.65,9.8,5,Red
3,11.2,0.28,0.56,1.9,0.075,17.0,60.0,0.9980,3.16,0.58,9.8,6,Red
4,7.4,0.70,0.00,1.9,0.076,11.0,34.0,0.9978,3.51,0.56,9.4,5,Red


# Split the data

Now we want to split the data into a training set and a test set for modeling. First we split for multiple linear regression with `alcohol` as the response, and rest of the variables as the predictors. We need to define X (explanatory variables) and Y (response variable) first, then use stratified sampling to split. Here, I split the data into 80% train, 20% test.

In [5]:
X_reg = wine.drop(columns=['alcohol'])
y_reg = wine['alcohol']
X_reg = pd.get_dummies(X_reg, columns=['Type'], drop_first=True)


X_reg_train, X_reg_test, y_reg_train, y_reg_test = train_test_split(
    X_reg, y_reg,
    test_size=0.2,
    random_state=42,
    stratify=wine['Type']
)


Then, we will split for logistic regression with same idea. We’ll use the type of wine as the response variable


In [6]:
y_log = wine['Type'].map({'Red': 0, 'White': 1})
X_log = wine.drop(columns=['Type'])

X_log_train, X_log_test, y_log_train, y_log_test = train_test_split(
    X_log, y_log,
    test_size=0.2,
    random_state=42,
    stratify=y_log
)

# Regression: Predict `alcohol`

Now we'll fit different types of regression and find out which one fit the best.

## Multiple Linear Regression 1: base fit with no interaction terms (additive model)


In [15]:
m1 = LinearRegression()
m1.fit(X_reg_train, y_reg_train)
print("Slopes = ", m1.coef_, "\n\n", "Intercept = ", m1.intercept_)

Slopes =  [ 5.09489671e-01  8.29707960e-01  5.35677764e-01  2.26028675e-01
 -1.07363016e+00 -3.12330120e-03 -2.86286760e-04 -6.47620140e+02
  2.58142483e+00  1.02456648e+00  1.01152976e-01 -1.09876867e+00] 

 Intercept =  640.8929137107565


## MLR 2: with interaction

In [18]:
poly_int = PolynomialFeatures(degree=2, interaction_only=True, include_bias=False)
X_train_int = poly_int.fit_transform(X_reg_train)
X_test_int = poly_int.transform(X_reg_test)

m2 = LinearRegression()
m2.fit(X_train_int, y_reg_train)

print("Slopes = ", m2.coef_, "\n\n", "Intercept = ", m2.intercept_)

Slopes =  [-2.45240140e+01 -1.80775923e+02 -1.76358131e+02 -3.25363499e+00
 -1.74464018e+02 -7.16954360e-01 -4.95430954e-01 -1.49429906e+03
 -2.00266999e+02 -3.68041177e+01  3.63311157e+01  6.96048519e+01
  3.28996352e-01  4.56112892e-02 -7.29999331e-03  1.40977807e-02
 -2.45377552e-03  8.18956420e-04  2.47095370e+01  5.18104918e-02
 -2.18230439e-01  3.59126923e-02  1.17874896e-01 -6.94970450e-01
 -2.23107921e-02 -7.13615435e+00  8.40453208e-03 -9.36641548e-03
  1.73622887e+02  1.38641657e+00  1.68306975e+00  1.92290808e-01
  1.81456452e+00 -8.74040128e-02 -4.24200763e+00 -1.04269021e-02
 -1.74732313e-03  1.76713154e+02  1.12049934e-01  7.42238387e-01
  1.72875472e-01  5.79826195e-01 -4.23918258e-02 -2.94073010e-04
 -4.31499069e-04  3.77057568e+00 -6.60802567e-02 -3.36006594e-02
  2.05107846e-02  4.29436817e-03 -6.34530368e-02  2.47659598e-02
  1.95816366e+02 -3.81053684e+00 -6.16704556e+00  2.82899447e-01
 -3.46824682e+00  1.47584986e-05  8.02063464e-01 -1.70908276e-02
 -1.32546947e-0

## MLR 3: Add Polynomial Term

In [20]:
poly = PolynomialFeatures(degree=2, include_bias=False)
X_train_poly = poly.fit_transform(X_reg_train)
X_test_poly = poly.transform(X_reg_test)

m3 = LinearRegression()
m3.fit(X_train_poly, y_reg_train)

print("Slopes = ", m3.coef_, "\n\n", "Intercept = ", m3.intercept_)

Slopes =  [ 2.02983457e+01 -1.03846835e+02 -3.88081642e+01  3.46627572e+01
  1.08326900e+03 -1.67298363e+00  3.96776954e-01 -9.29110868e+04
  7.12789689e+01  8.70115529e+01  2.80430520e+00 -9.61066116e+01
 -2.53836809e-02  4.16627489e-01  4.16690875e-01  8.74977565e-03
  2.71814193e-01 -3.01907040e-03  1.41819844e-03 -1.99069648e+01
  7.09329166e-03 -1.13592387e-01  2.19583273e-02 -1.53100174e-01
 -2.56884580e-01 -1.18835404e+00 -4.47140703e-02 -7.38869918e+00
  1.24032527e-02 -6.48441332e-03  9.71341926e+01  1.29417045e+00
  1.02796431e+00  1.16051101e-01  1.14194092e+00 -5.19802614e-01
 -3.63555644e-02 -4.04169474e-01 -7.25927191e-03 -2.25737088e-03
  3.18881279e+01  1.41280812e+00  5.54837043e-01  9.72518264e-02
  8.64335328e-01  4.97554490e-03  4.75332869e-01 -6.71308960e-04
  1.33280284e-04 -3.47161697e+01  2.63889879e-02 -4.19042089e-03
  2.11055018e-03 -1.25530846e-01  2.09362231e+00 -8.15945215e-02
  4.67597435e-02 -1.08734666e+03  1.38521813e+00 -5.95450478e+00
  4.75931651e-0

## MLR 4: standardized predictors

We'll compute the mean and standard deviatio for training set, and apply z score transformation.

from math import sqrt
temp = np.array([0, 3, 6, 10, 11])
print(temp.mean(), temp.std(ddof = 1))
(temp-temp.mean())/temp.std(ddof = 1)

In [25]:
X_reg_train_scaled = X_reg_train.apply(lambda x: (x-np.mean(x))/np.std(x), axis = 0)

m4 = LinearRegression()
m4.fit(X_reg_train_scaled, y_reg_train)

print("Slopes = ", m4.coef_, "\n\n", "Intercept = ", m4.intercept_)

Slopes =  [ 0.66282268  0.13659555  0.07698578  1.06804974 -0.03822332 -0.05550545
 -0.01624484 -1.95431401  0.41337476  0.15157121  0.08870369 -0.47328315] 

 Intercept =  10.489583092809905


## Compare MLR Models

We'll use CV and MSE scoring method to select the best MLR model.

In [27]:
cv_m1 = cross_validate(
    m1,
    X_reg_train,
    y_reg_train,
    cv=5,
    scoring="neg_mean_squared_error"
)

cv_m2 = cross_validate(
    m2,
    X_train_int,
    y_reg_train,
    cv=5,
    scoring="neg_mean_squared_error"
)

cv_m3 = cross_validate(
    m3,
    X_train_poly,
    y_reg_train,
    cv=5,
    scoring="neg_mean_squared_error"
)

cv_m4 = cross_validate(
    m4,
    X_reg_train_scaled,
    y_reg_train,
    cv=5,
    scoring="neg_mean_squared_error"
)

rmse1 = np.sqrt(-cv_m1["test_score"]).mean()
rmse2 = np.sqrt(-cv_m2["test_score"]).mean()
rmse3 = np.sqrt(-cv_m3["test_score"]).mean()
rmse4 = np.sqrt(-cv_m4["test_score"]).mean()

cv_results = pd.DataFrame({
    "Model": ["m1", "m2", "m3", "m4"],
    "CV_RMSE": [rmse1, rmse2, rmse3, rmse4]
})

print(cv_results)
print("Best model:", cv_results.loc[cv_results["CV_RMSE"].idxmin(), "Model"])

  Model   CV_RMSE
0    m1  0.508643
1    m2  0.461452
2    m3  0.417904
3    m4  0.508643
Best model: m3


## Fit a LASSO model
To fit a LASSO model, let's pick some predictor. We'll use fixed acidity, volatile acidity,	citric acid, free sulfur dioxide, pH, and residual sugar. For regularized models, we need use the standardized train set!

In [34]:
X_reg_train_sub = X_reg_train_scaled[["fixed acidity", "volatile acidity", "citric acid", "residual sugar", "free sulfur dioxide", "pH"]]

lasso_mod = LassoCV(cv=5, random_state=0) \
    .fit(X_reg_train_sub,
         y_reg_train)

Look at the tuning parameters and CV errors:

In [30]:
np.set_printoptions(suppress = True)
fit_info = np.array(list(zip(lasso_mod.alphas_, np.mean(lasso_mod.mse_path_, axis = 1))))
fit_info[fit_info[:,1].argsort()][0:10,].round(4)

array([[0.0004, 1.1974],
       [0.0005, 1.1974],
       [0.0005, 1.1974],
       [0.0005, 1.1974],
       [0.0006, 1.1974],
       [0.0006, 1.1974],
       [0.0006, 1.1974],
       [0.0007, 1.1974],
       [0.0007, 1.1974],
       [0.0008, 1.1974]])

In [32]:
print("alpha = ", lasso_mod.alpha_)
print("intercept = ",lasso_mod.intercept_)
print("slopes = ", np.array(list(zip(X_reg_train_sub.columns, lasso_mod.coef_))))

alpha =  0.0004272214779233313
intercept =  10.489583092809953
slopes =  [['fixed acidity' '-0.20010151152534975']
 ['volatile acidity' '-0.05962439764979632']
 ['citric acid' '0.10116200791391376']
 ['residual sugar' '-0.47488636233833326']]


Fit the best model:

In [35]:
lasso_best = Lasso(lasso_mod.alpha_).fit(X_reg_train_sub,y_reg_train)

## Ridge Regression
 Let's use the same 6 predictors to fit the Ridge Regression.

In [36]:
ridge = RidgeCV(alphas=np.logspace(-3,3,50), cv=5)
ridge.fit(X_reg_train_sub, y_reg_train)

RidgeCV(alphas=array([   0.001     ,    0.00132571,    0.00175751,    0.00232995,
          0.00308884,    0.00409492,    0.00542868,    0.00719686,
          0.00954095,    0.01264855,    0.01676833,    0.02222996,
          0.02947052,    0.0390694 ,    0.05179475,    0.06866488,
          0.09102982,    0.12067926,    0.15998587,    0.21209509,
          0.28117687,    0.37275937,    0.49417134,    0.65512856,
          0.86851137,    1.1513954 ,    1.52641797,    2.02358965,
          2.6826958 ,    3.55648031,    4.71486636,    6.25055193,
          8.28642773,   10.98541142,   14.56348478,   19.30697729,
         25.59547923,   33.93221772,   44.98432669,   59.63623317,
         79.06043211,  104.81131342,  138.94954944,  184.20699693,
        244.20530945,  323.74575428,  429.19342601,  568.9866029 ,
        754.31200634, 1000.        ]),
        cv=5)

## Elastic Net

In [40]:


regr = ElasticNetCV(cv=5,
 random_state=0,
 l1_ratio = [0.1, 0.25, 0.5, 0.75, 0.9, 0.95, 0.96, 0.98, 0.99, 1],
 n_alphas = 50)

regr.fit(X_reg_train_sub, y_reg_train)

print("alpha = ", regr.alpha_)

print("ratio = ", regr.l1_ratio_)

print("Slopes = ", np.array(list(zip(X_reg_train_sub.columns, regr.coef_))))


alpha =  0.0065211853986533
ratio =  0.1
Slopes =  [['fixed acidity' '-0.22669252606593396']
 ['volatile acidity' '-0.09237925904875119']
 ['citric acid' '0.11159067459407815']
 ['residual sugar' '-0.4209300591097273']
 ['free sulfur dioxide' '-0.14354146983819432']
 ['pH' '0.010980333830399784']]


## Compare on Test sets

First, standardize test sets using training set

In [50]:
means = X_reg_train.mean()
stds = X_reg_train.std()

cols = ["fixed acidity", "volatile acidity", "citric acid",
        "residual sugar", "free sulfur dioxide", "pH"]

X_reg_test_scaled = X_reg_test[cols].copy()

def my_std_fun(x, mean, std):
    return (x - mean) / std

for x in cols:
    X_reg_test_scaled[x] = my_std_fun(X_reg_test[x], means[x], stds[x])

X_reg_test_scaled.head()

,fixed acidity,volatile acidity,citric acid,residual sugar,free sulfur dioxide,pH
1739,-0.703905,-0.174740,0.150435,-0.675471,-0.592160,0.512172
2845,0.602703,-0.842838,0.567883,-0.040650,0.589400,-0.237127
5282,-0.319609,-1.146518,0.637458,1.588725,2.755593,0.137523
5822,-1.472498,0.554093,-1.449783,-0.908239,-1.379867,1.823445
5237,-0.627046,-0.296213,0.011286,-0.633150,0.195546,0.137523


In [56]:
# predictions
m3_pred = m3.predict(X_test_poly)
ridge_pred = ridge.predict(X_reg_test_scaled)
lasso_pred = lasso_best.predict(X_reg_test_scaled)
regr_pred = regr.predict(X_reg_test_scaled)

# RMSE
print(
    "MLR3 RMSE:", np.sqrt(mean_squared_error(y_reg_test, m3_pred)), "\n",
    "Ridge RMSE:", np.sqrt(mean_squared_error(y_reg_test, ridge_pred)),"\n",
    "LASSO RMSE:", np.sqrt(mean_squared_error(y_reg_test, lasso_pred)),"\n",
    "Elastic Net RMSE:", np.sqrt(mean_squared_error(y_reg_test, regr_pred)), "\n"

)

#MAE
print(
    "MLR3 MAE:", mean_absolute_error(y_reg_test, m3_pred), "\n",
    "Ridge MAE:", mean_absolute_error(y_reg_test, ridge_pred), "\n",
    "LASSO MAE:", mean_absolute_error(y_reg_test, lasso_pred), "\n",
    "Elastic Net MAE:", mean_absolute_error(y_reg_test, regr_pred)
)

MLR3 RMSE: 0.44330994024458026 
 Ridge RMSE: 1.085714586588647 
 LASSO RMSE: 1.0859929082568776 
 Elastic Net RMSE: 1.0858147782819667 

MLR3 MAE: 0.30695973964437934 
 Ridge MAE: 0.8734915910831565 
 LASSO MAE: 0.8731838018210967 
 Elastic Net MAE: 0.8733664759993907


Comparing RMSE and MAE, the polynomial multiple linear regression model (MLR3) achieved the best performance on the test set. Ridge, LASSO, and Elastic Net models all showed higher error. This suggests that including polynomial terms allowed the model to better capture nonlinear relationships in the data.

#Logistic Regression: Predict `type`

Let's now repeat the procedure to predict the type of wine.

## Logistic Regression 1: base

In [62]:
log1 = LogisticRegression(max_iter=5000)
print( "Slopes = ", log1.fit(X_log_train, y_log_train).coef_, "\n\n", "Intercept = ", log1.intercept_)

Slopes =  [[-1.12236186 -7.83989116  0.78107268  0.09209339 -2.64917157 -0.04501233
   0.05672966 -0.10956083 -5.90943152 -5.93082961  0.50632542  0.0660784 ]] 

 Intercept =  [24.781423]


## Logistic Regression 2: subset

Let's fit a subset of the predictor. I chose to fit fixed acidity, volatile acidity, density, pH and alcohol.

In [65]:
cols = ["fixed acidity", "volatile acidity", "density", "pH", "alcohol"]

X_log_train_m2 = X_log_train[cols]
X_log_test_m2 = X_log_test[cols]

log2 = LogisticRegression(max_iter=5000)

print( "Slopes = ", log2.fit(X_log_train_m2, y_log_train).coef_, "\n\n", "Intercept = ", log2.intercept_)

Slopes =  [[ -1.71408527 -10.68206346  -0.23836884  -9.59230027  -0.0823613 ]] 

 Intercept =  [50.22869947]


## Logistic Regression 3: Add interaction

In [71]:
poly_log_int = PolynomialFeatures(degree=2, interaction_only=True, include_bias=False)
X_train_m3 = poly_log_int.fit_transform(X_log_train)
X_test_m3 = poly_log_int.transform(X_log_test)

log3 = LogisticRegression(max_iter=10000, solver = "newton-cg",)

print( "Slopes = ", log3.fit(X_train_m3, y_log_train).coef_, "\n\n", "Intercept = ", log3.intercept_)

Slopes =  [[ 0.2881874  -0.22670168  0.21639389  0.67091251 -0.16717272  0.69546178
   2.29557856  0.04354711 -0.15291135 -0.19976376  1.00966747  0.26340195
  -1.31920623  0.67564517  0.10464809 -1.64280821 -0.02879868  0.02457473
  -0.15689231 -1.38657285 -1.53331943  0.29154789 -0.15013965  0.13535235
   0.72690568 -0.15615271 -0.03696608  0.00138391 -0.25374678 -1.36806251
  -0.2985586   1.17390823 -1.80880107 -0.41444295  0.08926454  0.2190241
  -0.07285255  0.19921585  0.5685584  -0.06848831 -0.2547185   0.08629212
   0.08207962  0.00216104 -0.00092091  0.6692078  -0.55543164 -0.14675415
  -0.05540376  0.08927059  0.18452966  0.05487359 -0.17022594 -0.69425367
  -0.17400661 -2.2265038  -0.84056594 -0.00005039 -0.59504799 -0.04370709
  -0.10282076  0.00688834  0.01931278 -2.81063244  0.15058038  0.10260384
  -0.00826629 -0.01006681 -0.35153075 -0.23765957  0.35521594 -0.06805179
  -1.06032241 -0.95014124  0.32569533 -0.31558561  0.57773271  0.03905482]] 

 Intercept =  [36.1963673

LR 4: Standardized predictors

In [73]:
X_log_train_scaled = X_log_train.apply(lambda x: (x-np.mean(x))/np.std(x), axis = 0)

lr4 = LogisticRegression(max_iter=5000)
lr4.fit(X_log_train_scaled, y_log_train)

print("Slopes = ",lr4.coef_, "\n\n", "Intercept = ", lr4.intercept_)

Slopes =  [[-0.30662326 -1.14633908  0.32861954  2.91896598 -0.89641829 -0.70495981
   2.65998294 -3.41268129 -0.35887058 -0.65166652 -1.09129941 -0.2720649 ]] 

 Intercept =  [3.75976747]


## Compare Logistic regression models


In [74]:
cv_log1 = cross_validate(
    log1,
    X_log_train, y_log_train,
    cv=5,
    scoring="neg_log_loss"
)

cv_log2 = cross_validate(
    log2,
    X_log_train_m2, y_log_train,
    cv=5,
    scoring="neg_log_loss"
)

cv_log3 = cross_validate(
    log3,
    X_train_m3, y_log_train,
    cv=5,
    scoring="neg_log_loss"
)

cv_log4 = cross_validate(
    lr4,
    X_log_train_scaled, y_log_train,
    cv=5,
    scoring="neg_log_loss"
)

log_loss1 = np.sqrt(-cv_log1["test_score"]).mean()
log_loss2 = np.sqrt(-cv_log2["test_score"]).mean()
log_loss3 = np.sqrt(-cv_log3["test_score"]).mean()
log_loss4 = np.sqrt(-cv_log4["test_score"]).mean()

cv_results = pd.DataFrame({
    "Model": ["log1", "log2", "log3", "log4"],
    "CV_log_loss": [log_loss1, log_loss2, log_loss3, log_loss4]
})

print(cv_results)
print("Best model:", cv_results.loc[cv_results["CV_log_loss"].idxmin(), "Model"])

  Model  CV_log_loss
0  log1     0.252535
1  log2     0.407886
2  log3     0.219542
3  log4     0.206793
Best model: log4


Logistic regression with standardize predictors is the best model!

## LASSO Logistic Regression
I used same idea and the same subset as for regular MLR LASSO model. LASSO model uses `l1` penalty.

In [83]:
X_log_train_sub = X_log_train_scaled[["fixed acidity", "volatile acidity", "citric acid", "residual sugar", "free sulfur dioxide", "pH"]]

lasso_log = LogisticRegressionCV(cv = 5,
                                 solver = "saga",
                                 penalty = "l1",
                                 Cs = 250,
                                 scoring = "neg_log_loss",
                                 random_state = 10)

Look at tuning parameter:

In [84]:
lasso_log.fit(X_log_train_sub, y_log_train)

print("Best C:", lasso_log.C_)

Best C: [0.45988644]


Fit optimal model

In [101]:
lasso_log_best= LogisticRegression(solver = "saga",
                                   penalty = "l1",
                                   C = lasso_log.C_[0],
                                   random_state = 5)

lasso_log_best.fit(X_log_train_sub, y_log_train)

LogisticRegression(C=np.float64(0.4598864379394479), penalty='l1',
                   random_state=5, solver='saga')

## Ridge Logistic Regression

Now we'll fit the ridge logistic regression using `l2` penalty.

In [86]:
ridge_log = LogisticRegressionCV(
    cv=5,
    penalty="l2",
    solver="lbfgs",
    Cs=250,
    scoring="neg_log_loss",
    random_state=10,
    max_iter=5000
)

ridge_log.fit(X_log_train_sub, y_log_train)

print("Best C:", ridge_log.C_)

Best C: [0.61825076]


Fit the optimal model with the best C

In [104]:
ridge_reg_best= LogisticRegression(solver = "lbfgs",
                                   penalty = "l2",
                                   C = ridge_log.C_[0],
                                   random_state = 5)

ridge_reg_best.fit(X_log_train_sub, y_log_train)



LogisticRegression(C=np.float64(0.6182507596208566), random_state=5)

##Elastic Net

Finally we fit the elastic net logistic regression model with `elasticnet` penalty.

In [92]:
en_log = LogisticRegressionCV(
    cv=5,
    penalty="elasticnet",
    solver="saga",
    l1_ratios=[0.2, 0.5, 0.8],
    Cs=250,
    scoring="neg_log_loss",
    random_state=10,
    max_iter=5000
)

en_log.fit(X_log_train_sub, y_log_train)

print("Best C:", en_log.C_)
print("Best l1_ratio:", en_log.l1_ratio_)

Best C: [0.57416425]
Best l1_ratio: [0.2]


Optimal model:

In [105]:
en_best= LogisticRegression(solver = "saga",
                                   penalty = "elasticnet",
                                   C = en_log.C_[0],
                                  l1_ratio = en_log.l1_ratio_,
                                   random_state = 5)

ridge_reg_best.fit(X_log_train_sub, y_log_train)



LogisticRegression(C=np.float64(0.6182507596208566), random_state=5)

##Compare Logistic Regression models
First as usual, standardize test set.

In [93]:
means = X_log_train.mean()
stds = X_log_train.std()

cols = ["fixed acidity", "volatile acidity", "citric acid",
        "residual sugar", "free sulfur dioxide", "pH"]

X_log_test_scaled = X_log_test[cols].copy()

def my_std_fun(x, mean, std):
    return (x - mean) / std

for x in cols:
    X_log_test_scaled[x] = my_std_fun(X_log_test[x], means[x], stds[x])

X_log_test_scaled.head()

,fixed acidity,volatile acidity,citric acid,residual sugar,free sulfur dioxide,pH
1739,-0.703905,-0.174740,0.150435,-0.675471,-0.592160,0.512172
2845,0.602703,-0.842838,0.567883,-0.040650,0.589400,-0.237127
5282,-0.319609,-1.146518,0.637458,1.588725,2.755593,0.137523
5822,-1.472498,0.554093,-1.449783,-0.908239,-1.379867,1.823445
5237,-0.627046,-0.296213,0.011286,-0.633150,0.195546,0.137523


In [96]:
X_log_test_scaled_full = X_log_test.copy()

def my_std_fun(x, mean, std):
    return (x - mean) / std

for x in cols:
    X_log_test_scaled_full[x] = my_std_fun(X_log_test[x], means[x], stds[x])

X_log_test_scaled_full.head()

,fixed acidity,volatile acidity,citric acid,residual sugar,chlorides,free sulfur dioxide,total sulfur dioxide,density,pH,sulphates,alcohol,quality
1739,-0.703905,-0.174740,0.150435,-0.675471,0.045,-0.592160,77.0,0.99270,0.512172,0.43,10.2,5
2845,0.602703,-0.842838,0.567883,-0.040650,0.055,0.589400,167.0,0.99530,-0.237127,0.40,10.6,7
5282,-0.319609,-1.146518,0.637458,1.588725,0.044,2.755593,183.0,0.99742,0.137523,0.78,10.2,6
5822,-1.472498,0.554093,-1.449783,-0.908239,0.029,-1.379867,51.0,0.99076,1.823445,0.48,11.2,4
5237,-0.627046,-0.296213,0.011286,-0.633150,0.014,0.195546,89.0,0.99008,0.137523,0.66,12.5,7


In [106]:
# probabilities
log4_prob = lr4.predict_proba(X_log_test_scaled_full)[:, 1]
lasso_prob = lasso_log_best.predict_proba(X_log_test_scaled)[:, 1]
ridge_prob = ridge_reg_best.predict_proba(X_log_test_scaled)[:, 1]
en_prob = en_log.predict_proba(X_log_test_scaled)[:, 1]

log4_preds = lr4.predict(X_log_test_scaled_full)
lasso_log_preds = lasso_log_best.predict(X_log_test_scaled)
ridge_log_preds = ridge_reg_best.predict(X_log_test_scaled)
en_log_preds = en_log.predict(X_log_test_scaled)

from sklearn.metrics import log_loss, accuracy_score

# log loss
print(
    "LR4 log loss:", log_loss(y_log_test, log4_prob), "\n",
    "Ridge log loss:", log_loss(y_log_test, ridge_prob), "\n",
    "LASSO log loss:", log_loss(y_log_test, lasso_prob), "\n",
    "Elastic Net log loss:", log_loss(y_log_test, en_prob), "\n"
)

#MAE
print(
    "LR4 accuracy:", accuracy_score(y_log_test, log4_preds), "\n",
    "Ridge accuracy:", accuracy_score(y_log_test, lasso_log_preds),"\n",
    "LASSO accuracy:", accuracy_score(y_log_test, ridge_log_preds),"\n",
    "Elastic Net accuracy:", accuracy_score(y_log_test, en_log_preds), "\n"

)

LR4 log loss: 8.114651304672465 
 Ridge log loss: 0.11164411871462383 
 LASSO log loss: 0.11152733950812807 
 Elastic Net log loss: 0.11163584441049426 

LR4 accuracy: 0.7538461538461538 
 Ridge accuracy: 0.9615384615384616 
 LASSO accuracy: 0.9607692307692308 
 Elastic Net accuracy: 0.9607692307692308 



The standardized logistic regression model performed poorly, with a very high log-loss and lower accuracy. In contrast, the regularized models all achieved substantially better performance. Among the regularized models, performance differences were minimal, although LASSO achieved the lowest log-loss.